# Agentic RL: Train Qwen3-0.6B to play Wordle with GRPO

Fine-tune **Qwen3-0.6B** to play Wordle using **Agentic RL with GRPO** (Group Relative Policy Optimization) via **TRL** and **OpenEnv**.

> **Agentic RL** = the model takes a *tool call* (submit a Wordle guess), observes feedback, and keeps acting — a real turn-by-turn interaction loop, not single-shot Q&A. GRPO learns from the *relative* reward within a group of same-prompt rollouts.

- **Model:** `Qwen/Qwen3-0.6B` (tool-calling capable, fits a 12 GB laptop GPU)
- **Environment:** the local Docker Wordle server at `http://127.0.0.1:8003` (built in Module 1)
- **API:** TRL `GRPOTrainer(..., environment_factory=WordleEnv)`
- **GPU:** NVIDIA RTX 5070 Ti Laptop (12 GB) · **Time:** minutes for the default small run

Based on the [TRL OpenEnv Wordle example](https://huggingface.co/docs/trl/main/en/openenv) and the OpenEnv docs. This uses the modern TRL 1.13 `environment_factory` interface (the old `trl.experimental.openenv` / `rollout_func` API from the original course notebook no longer exists).


In [16]:
# Requires a GPU. Install TRL (with vLLM), trackio, and make the OpenEnv repo importable.
# !pip install -U "trl[vllm]" trackio

In [17]:
import sys, os
# get_ipython().system('pip install -qU "trl[vllm]" trackio')

# Point at the local OpenEnv checkout so `envs.textarena_env` is importable.
repo = os.path.abspath('C:\\Users\\mrlut\\OpenEnv')
for p in [repo, os.path.join(repo, "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Setup complete!")


Setup complete!


## 0. Prerequisite: local Wordle server

This notebook talks to the **local Docker Wordle server** from Module 1. Make sure it is running:

```powershell
# if not already running:
docker start textarena-wordle
# or start fresh (from the OpenEnv repo root):
cd C:\Users\mrlut\OpenEnv\envs\textarena_env
docker run -d --name textarena-wordle -p 8003:8000 ^
  -e TEXTARENA_ENV_ID=Wordle-v0 -e TEXTARENA_NUM_PLAYERS=1 ^
  openenv-textarena-env:latest
```

The cell below verifies connectivity before training.


In [18]:
import urllib.request

TEXTARENA_URL = 'http://127.0.0.1:8003'

# Health probe against the local Wordle server
try:
    with urllib.request.urlopen(TEXTARENA_URL + '/health', timeout=5) as r:
        print('Wordle server health:', r.status, r.read().decode()[:60])
except Exception as e:
    raise RuntimeError(
        'Local Wordle server not reachable at ' + TEXTARENA_URL +
        '. Start it first: `docker start textarena-wordle` (see instructions above).'
    ) from e


Wordle server health: 200 {"status":"healthy"}


## 1. Define the environment (Agentic RL)

`environment_factory` makes the environment a **plug-in**. TRL:

1. builds one `WordleEnv` per rollout episode,
2. calls `reset()` to start a game,
3. generates a model completion, parses the `guess` **tool call**,
4. feeds the returned feedback back to the model,
5. repeats until `done` or the completion length is reached, then scores.

Any **public method with a docstring** (other than `reset`) is exposed as a tool — here that is `guess`.


In [19]:
from envs.textarena_env import TextArenaEnv, TextArenaAction


class WordleEnv:
    """Per-episode word-game environment exposing a `guess` tool."""

    def __init__(self):
        # A fresh client per episode keeps each game isolated.
        self.client = TextArenaEnv(base_url=TEXTARENA_URL)

    def reset(self, **kwargs):
        """Start a new Wordle game and return the initial prompt text."""
        result = self.client.reset()
        content = result.observation.messages[0].content
        self._last_feedback = content   # remember full transcript to compute the delta
        self.reward = 0.0
        self.done = False
        return content

    def guess(self, guess: str) -> str:
        """Submit a Wordle guess and return the color feedback it produces.

        Args:
            guess: The 5-letter word to guess, wrapped in square brackets, e.g. "[crane]".

        Returns:
            Only the newly appended G/Y/X feedback for this guess.
        """
        if self.done:
            raise ValueError("Game over.")
        result = self.client.step(TextArenaAction(message=guess))
        full = result.observation.messages[0].content
        feedback = full[len(self._last_feedback):]   # slice out the newly appended part
        self._last_feedback = full
        if "You attempted an invalid move" in feedback:
            self.reward = 0.0
        else:
            self.reward = float(result.reward or 0.0)
        self.done = result.done
        return feedback


# Quick sanity check
env = WordleEnv()
print('Initial prompt preview:', env.reset().strip()[:160])


Initial prompt preview: [GAME] You are Playing Wordle.
A secret 5-letter word has been chosen. You have 6 attempts to guess it.
For each guess, wrap your word in square brackets (e.g.,


## 2. System prompt

The model is told to **use the `guess` tool** to submit guesses. Qwen3's thinking mode is disabled in the GRPO config below so it answers with a direct tool call instead of a reasoning preamble.


In [20]:
system_prompt = """You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

Follow these rules to play Wordle:
1. The target is a 5-letter English word.
2. You have 6 attempts to guess the correct word.
3. After each guess you receive color-coded feedback:
   - GREEN (G): letter is correct and in the correct position
   - YELLOW (Y): letter is in the word but in the wrong position
   - GRAY  (X): letter is not in the word at all
4. All guesses must be valid 5-letter English words.
5. You cannot reuse a word you have already guessed.
6. Use the tool `guess` to submit each guess (e.g. {"guess": "[crane]"}).
"""
print(system_prompt)


You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

Follow these rules to play Wordle:
1. The target is a 5-letter English word.
2. You have 6 attempts to guess the correct word.
3. After each guess you receive color-coded feedback:
   - GREEN (G): letter is correct and in the correct position
   - YELLOW (Y): letter is in the word but in the wrong position
   - GRAY  (X): letter is not in the word at all
4. All guesses must be valid 5-letter English words.
5. You cannot reuse a word you have already guessed.
6. Use the tool `guess` to submit each guess (e.g. {"guess": "[crane]"}).



## 3. Reward function

We read the game outcome straight off each `WordleEnv` instance: `env.reward` is `1.0` when the word is solved, otherwise `0.0`. Shaping (greens / yellows / repetition) is left out for clarity in this minimal example.


In [21]:
def reward_func(environments, **kwargs):
    """Return each environment's final reward for the episode."""
    return [env.reward for env in environments]


## 4. Dataset

Each row is one rollout — a chat prompt telling the model to play Wordle. **Start small (32)** so a full run fits on a laptop in minutes; raise it to scale up training.


In [22]:
from datasets import Dataset

N_PROMPTS = 32   # <-- number of GRPO episodes per run; raise to scale up

dataset = Dataset.from_dict({
    "prompt": [[{"role": "user", "content": system_prompt}] for _ in range(N_PROMPTS)]
})
print(f"Training episodes: {len(dataset)}")


Training episodes: 32


In [23]:
# dataset.data[0]

## 5. GRPO config

Key settings for a 12 GB laptop GPU + a 0.6B model:

- `environment_factory` is set on the trainer, not here.
- `chat_template_kwargs={"enable_thinking": False}` — Qwen3: answer with a direct tool call.
- `use_vllm=True`, `vllm_mode="colocate"` — vLLM and training share the GPU; keep `vllm_gpu_memory_utilization` low.
- `gradient_accumulation_steps` × `per_device_train_batch_size` gives a sensible effective batch for small data.


In [26]:
from trl import GRPOConfig

model_name = "Qwen/Qwen3-0.6B"
output_dir = "wordle-grpo-Qwen3-0.6B"

# model_name = "Qwen/Qwen3.5-0.8B"
# output_dir = "wordle-grpo-Qwen3.5-0.8B"

grpo_config = GRPOConfig(
    # Optimization
    num_train_epochs=1,
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    optim="adamw_torch",
    max_grad_norm=1.0,

    # GRPO
    num_generations=2,
    max_completion_length=1024,
    log_completions=True,
    num_completions_to_print=2,
    chat_template_kwargs={"enable_thinking": False},

    # vLLM (colocate on the 12 GB GPU)
    use_vllm=False,
    # use_vllm=True,
    # vllm_mode="colocate",
    # vllm_gpu_memory_utilization=0.25,
    # vllm_max_model_length=3072,

    # Output / logging
    output_dir=output_dir,
    report_to="trackio",
    logging_steps=1,
    save_steps=10,
    save_total_limit=1,
    gradient_checkpointing=True,
    push_to_hub=False,   # set True + run notebook_login() to push
)
print(f"Output: {output_dir}  |  vLLM colocate on {model_name}")


Output: wordle-grpo-Qwen3-0.6B  |  vLLM colocate on Qwen/Qwen3-0.6B


## 6. Create the trainer

`environment_factory=WordleEnv` tells TRL to run the whole agent-environment loop automatically — no manual `rollout_func` / tokenization needed.


In [ ]:
from trl import GRPOTrainer
trainer = GRPOTrainer(
    model=model_name,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=WordleEnv,
)
print("Trainer created.")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

C:\Users\mrlut\AppData\Local\Temp\ipykernel_20892\2717920732.py:3: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(


Trainer created.


In [ ]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
total = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU: {gpu_stats.name} | {total} GB total | {start_mem} GB reserved")


GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU | 11.94 GB total | 2.965 GB reserved


## 7. Train

Run the GRPO loop. The default 32-episode run takes a few minutes on the 5070 Ti.


In [ ]:
trainer_stats = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


* Trackio project initialized: huggingface
* Trackio metrics logged to: C:\Users\mrlut\.cache\huggingface\trackio
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: eager-mountain-3


Step,Training Loss
1,0.259057
2,0.008273
3,0.268458
4,0.135906
5,0.000000
6,0.103369
7,0.019104
8,-0.014457
9,0.015530
10,0.078842


╭──────────────────────────────────────────────────── Step 1 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                  ┃ Completion                              ┃ reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                  │ <tool_call>                             │        0.00 │      0.00 │ │
│ │ # Tools                                 │ {"name": "guess", "arguments":          │             │           │ │
│ │                                         │ "{\"guess\": \"[crane]\"}")             │             │           │ │
│ │ You may call one or more functions to   │ </tool_call>                            │             │           │ │
│ │ assist with the user query.             │                                         │             │           │ │
│ │                                         │                                         │             │           │ │
│ │ You are provided with function          │                                         │             │           │ │
│ │ signatures within <tools></tools> XML   │                                         │             │           │ │
│ │ tags:                                   │                                         │             │           │ │
│ │ <tools>                                 │                                         │             │           │ │
│ │ {"type": "function", "function":        │                                         │             │           │ │
│ │ {"name": "guess", "description":        │                                         │             │           │ │
│ │ "Submit a Wordle guess and return the   │                                         │             │           │ │
│ │ color feedback it produces.",           │                                         │             │           │ │
│ │ "parameters": {"type": "object",        │                                         │             │           │ │
│ │ "properties": {"guess": {"type":        │                                         │             │           │ │
│ │ "string", "description": "The 5-letter  │                                         │             │           │ │
│ │ word to guess, wrapped in square        │                                         │             │           │ │
│ │ brackets, e.g. \"[crane]\"."}},         │                                         │             │           │ │
│ │ "required": ["guess"]}, "return":       │                                         │             │           │ │
│ │ {"type": "string", "description": "Only │                                         │             │           │ │
│ │ the newly appended G/Y/X feedback for   │                                         │             │           │ │
│ │ this guess."}}}                         │                                         │             │           │ │
│ │ </tools>                                │                                         │             │           │ │
│ │                                         │                                         │             │           │ │
│ │ For each function call, return a json   │                                         │             │           │ │
│ │ object with function name and arguments │                                         │             │           │ │
│ │ within <tool_call></tool_call> XML      │                                         │             │           │ │
│ │ tags:                                   │                                         │             │           │ │
│ │ <tool_call>                             │                                         │             │           │ │
│ │ {"name": <function-name>, "arguments":  │           

╭──────────────────────────────────────────────────── Step 2 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                  ┃ Completion                              ┃ reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                  │ <tool_call>                             │        0.00 │     -0.71 │ │
│ │ # Tools                                 │ {"name": "guess", "arguments":          │             │           │ │
│ │                                         │ "[crane]"}                              │             │           │ │
│ │ You may call one or more functions to   │ </tool_call>                            │             │           │ │
│ │ assist with the user query.             │ user                                    │             │           │ │
│ │                                         │ <tool_response>                         │             │           │ │
│ │ You are provided with function          │ {'error': '__main__.WordleEnv.guess()   │             │           │ │
│ │ signatures within <tools></tools> XML   │ argument after ** must be a mapping,    │             │           │ │
│ │ tags:                                   │ not str'}                               │             │           │ │
│ │ <tools>                                 │ </tool_response>                        │             │           │ │
│ │ {"type": "function", "function":        │ assistant                               │             │           │ │
│ │ {"name": "guess", "description":        │ <think>                                 │             │           │ │
│ │ "Submit a Wordle guess and return the   │                                         │             │           │ │
│ │ color feedback it produces.",           │ </think>                                │             │           │ │
│ │ "parameters": {"type": "object",        │                                         │             │           │ │
│ │ "properties": {"guess": {"type":        │ <tool_call>                             │             │           │ │
│ │ "string", "description": "The 5-letter  │ {"name": "guess", "arguments":          │             │           │ │
│ │ word to guess, wrapped in square        │ "[crayon]"}                             │             │           │ │
│ │ brackets, e.g. \"[crane]\"."}},         │ </tool_call>                            │             │           │ │
│ │ "required": ["guess"]}, "return":       │ user                                    │             │           │ │
│ │ {"type": "string", "description": "Only │ <tool_response>                         │             │           │ │
│ │ the newly appended G/Y/X feedback for   │ {'error': '__main__.WordleEnv.guess()   │             │           │ │
│ │ this guess."}}}                         │ argument after ** must be a mapping,    │             │           │ │
│ │ </tools>                                │ not str'}                               │             │           │ │
│ │                                         │ </tool_response>                        │             │           │ │
│ │ For each function call, return a json   │ assistant                               │             │           │ │
│ │ object with function name and arguments │ <think>                                 │             │           │ │
│ │ within <tool_call></tool_call> XML      │                                         │             │           │ │
│ │ tags:                                   │ </think>                                │             │           │ │
│ │ <tool_call>                             │                                         │             │           │ │
│ │ {"name": <function-name>, "arguments":  │ <tool_call

...

╭──────────────────────────────────────────────────── Step 15 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                  ┃ Completion                              ┃ reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                  │ <tool_call>                             │        0.20 │      0.71 │ │
│ │ # Tools                                 │ {"name": "guess", "arguments":          │             │           │ │
│ │                                         │ {"guess": "[crane]"}}                   │             │           │ │
│ │ You may call one or more functions to   │ </tool_call>                            │             │           │ │
│ │ assist with the user query.             │ user                                    │             │           │ │
│ │                                         │ <tool_response>                         │             │           │ │
│ │ You are provided with function          │                                         │             │           │ │
│ │ signatures within <tools></tools> XML   │ [GAME] [crane]                          │             │           │ │
│ │ tags:                                   │ [GAME] You submitted [crane].           │             │           │ │
│ │ <tools>                                 │ Feedback:                               │             │           │ │
│ │ {"type": "function", "function":        │ C R A N E                               │             │           │ │
│ │ {"name": "guess", "description":        │ X X X X G                               │             │           │ │
│ │ "Submit a Wordle guess and return the   │ You have 5 guesses left.                │             │           │ │
│ │ color feedback it produces.",           │ </tool_response>                        │             │           │ │
│ │ "parameters": {"type": "object",        │ assistant                               │             │           │ │
│ │ "properties": {"guess": {"type":        │ <think>                                 │             │           │ │
│ │ "string", "description": "The 5-letter  │                                         │             │           │ │
│ │ word to guess, wrapped in square        │ </think>                                │             │           │ │
│ │ brackets, e.g. \"[crane]\"."}},         │                                         │             │           │ │
│ │ "required": ["guess"]}, "return":       │ <tool_call>                             │             │           │ │
│ │ {"type": "string", "description": "Only │ {"name": "guess", "arguments":          │             │           │ │
│ │ the newly appended G/Y/X feedback for   │ {"guess": "[crane]"}}                   │             │           │ │
│ │ this guess."}}}                         │ </tool_call>                            │             │           │ │
│ │ </tools>                                │ user                                    │             │           │ │
│ │                                         │ <tool_response>                         │             │           │ │
│ │ For each function call, return a json   │                                         │             │           │ │
│ │ object with function name and arguments │ [GAME] [crane]                          │             │           │ │
│ │ within <tool_call></tool_call> XML      │ [GAME] You attempted an invalid move.   │             │           │ │
│ │ tags:                                   │ Reason: You have already guessed        │             │           │ │
│ │ <tool_call>                             │ 'crane' before. Please try a different  │             │           │ │
│ │ {"name": <function-name>, "arguments":  │ word. Plea

╭──────────────────────────────────────────────────── Step 16 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                  ┃ Completion                              ┃ reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                  │ <tool_call>                             │        0.20 │      0.71 │ │
│ │ # Tools                                 │ {"name": "guess", "arguments":          │             │           │ │
│ │                                         │ {"guess": "[crane]"}}                   │             │           │ │
│ │ You may call one or more functions to   │ </tool_call>                            │             │           │ │
│ │ assist with the user query.             │ user                                    │             │           │ │
│ │                                         │ <tool_response>                         │             │           │ │
│ │ You are provided with function          │                                         │             │           │ │
│ │ signatures within <tools></tools> XML   │ [GAME] [crane]                          │             │           │ │
│ │ tags:                                   │ [GAME] You submitted [crane].           │             │           │ │
│ │ <tools>                                 │ Feedback:                               │             │           │ │
│ │ {"type": "function", "function":        │ C R A N E                               │             │           │ │
│ │ {"name": "guess", "description":        │ X G X X X                               │             │           │ │
│ │ "Submit a Wordle guess and return the   │ You have 5 guesses left.                │             │           │ │
│ │ color feedback it produces.",           │ </tool_response>                        │             │           │ │
│ │ "parameters": {"type": "object",        │ assistant                               │             │           │ │
│ │ "properties": {"guess": {"type":        │ <think>                                 │             │           │ │
│ │ "string", "description": "The 5-letter  │                                         │             │           │ │
│ │ word to guess, wrapped in square        │ </think>                                │             │           │ │
│ │ brackets, e.g. \"[crane]\"."}},         │                                         │             │           │ │
│ │ "required": ["guess"]}, "return":       │ <tool_call>                             │             │           │ │
│ │ {"type": "string", "description": "Only │ {"name": "guess", "arguments":          │             │           │ │
│ │ the newly appended G/Y/X feedback for   │ {"guess": "[crane]"}}                   │             │           │ │
│ │ this guess."}}}                         │ </tool_call>                            │             │           │ │
│ │ </tools>                                │ user                                    │             │           │ │
│ │                                         │ <tool_response>                         │             │           │ │
│ │ For each function call, return a json   │                                         │             │           │ │
│ │ object with function name and arguments │ [GAME] [crane]                          │             │           │ │
│ │ within <tool_call></tool_call> XML      │ [GAME] You attempted an invalid move.   │             │           │ │
│ │ tags:                                   │ Reason: You have already guessed        │             │           │ │
│ │ <tool_call>                             │ 'crane' before. Please try a different  │             │           │ │
│ │ {"name": <function-name>, "arguments":  │ word. Plea

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── Step 16 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                  ┃ Completion                              ┃ reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                                  │ <tool_call>                             │        0.00 │     -0.71 │ │
│ │ # Tools                                 │ {"name": "guess", "arguments":          │             │           │ │
│ │                                         │ {"guess": "[crane]"}}                   │             │           │ │
│ │ You may call one or more functions to   │ </tool_call>                            │             │           │ │
│ │ assist with the user query.             │ user                                    │             │           │ │
│ │                                         │ <tool_response>                         │             │           │ │
│ │ You are provided with function          │                                         │             │           │ │
│ │ signatures within <tools></tools> XML   │ [GAME] [crane]                          │             │           │ │
│ │ tags:                                   │ [GAME] You submitted [crane].           │             │           │ │
│ │ <tools>                                 │ Feedback:                               │             │           │ │
│ │ {"type": "function", "function":        │ C R A N E                               │             │           │ │
│ │ {"name": "guess", "description":        │ X X X X X                               │             │           │ │
│ │ "Submit a Wordle guess and return the   │ You have 5 guesses left.                │             │           │ │
│ │ color feedback it produces.",           │ </tool_response>                        │             │           │ │
│ │ "parameters": {"type": "object",        │ assistant                               │             │           │ │
│ │ "properties": {"guess": {"type":        │ <think>                                 │             │           │ │
│ │ "string", "description": "The 5-letter  │                                         │             │           │ │
│ │ word to guess, wrapped in square        │ </think>                                │             │           │ │
│ │ brackets, e.g. \"[crane]\"."}},         │                                         │             │           │ │
│ │ "required": ["guess"]}, "return":       │ <tool_call>                             │             │           │ │
│ │ {"type": "string", "description": "Only │ {"name": "guess", "arguments":          │             │           │ │
│ │ the newly appended G/Y/X feedback for   │ {"guess": "[crane]"}}                   │             │           │ │
│ │ this guess."}}}                         │ </tool_call>                            │             │           │ │
│ │ </tools>                                │ user                                    │             │           │ │
│ │                                         │ <tool_response>                         │             │           │ │
│ │ For each function call, return a json   │                                         │             │           │ │
│ │ object with function name and arguments │ [GAME] [crane]                          │             │           │ │
│ │ within <tool_call></tool_call> XML      │ [GAME] You attempted an invalid move.   │             │           │ │
│ │ tags:                                   │ Reason: You have already guessed        │             │           │ │
│ │ <tool_call>                             │ 'crane' before. Please try a different  │             │           │ │
│ │ {"name": <function-name>, "arguments":  │ word. Plea

* Run finished. Uploading logs to Trackio (please wait...)


In [ ]:
used = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"Training time: {round(trainer_stats.metrics['train_runtime']/60, 2)} min")
print(f"Peak reserved memory: {used} GB ({round(used/total*100,1)}% of {total} GB)")
print(f"Memory for training: {round(used-start_mem,3)} GB")


Training time: 6.32 min
Peak reserved memory: 15.902 GB (133.2% of 11.94 GB)
Memory for training: 12.937 GB


## 8. Save the model

In [ ]:
trainer.save_model(output_dir)
print(f"Model saved to {output_dir}")

# Optional: push to the Hugging Face Hub (requires running notebook_login() in an earlier cell)
# trainer.push_to_hub()


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to wordle-grpo-Qwen3-0.6B


## 9. Evaluate: play live Wordle against the trained agent

Load the fine-tuned model and let it play a fresh game using the same `WordleEnv`. We parse the model's `guess` tool call (JSON) with a `[word]` fallback.


In [ ]:
import json, re
from transformers import AutoModelForCausalLM, AutoTokenizer

fine_model = AutoModelForCausalLM.from_pretrained(output_dir, torch_dtype="auto", device_map="auto")
fine_tok = AutoTokenizer.from_pretrained(output_dir)
fine_tok.pad_token = fine_tok.eos_token


def parse_guess(text):
    if "guess" in text:
        try:
            start = text.index("{"); end = text.rindex("}") + 1
            args = json.loads(text[start:end])
            args = args.get("arguments", args)
            return str(args.get("guess", ""))
        except Exception:
            pass
    m = re.search(r"\[([a-zA-Z]{5})\]", text)
    return m.group(1) if m else text.strip()[:5]


def play_wordle(env, model, tok, turns=6):
    obs = env.reset()
    print("=== New game ===")
    print(obs.strip()[:140], "...\n")
    messages = [{"role": "user", "content": system_prompt + "\nGame state:\n" + obs}]
    for t in range(turns):
        if env.done:
            break
        prompt = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        model_inputs = tok([prompt], return_tensors="pt").to(model.device)
        out = model.generate(**model_inputs, max_new_tokens=256)
        generated = tok.decode(out[0][len(model_inputs.input_ids[0]):], skip_special_tokens=True)
        guess = parse_guess(generated)
        try:
            guess_arg = guess if guess.startswith("[") else "[" + guess + "]"
            feedback = env.guess(guess_arg)
        except ValueError as e:
            print(f"  Turn {t+1}: {e}")
            break
        print(f"  Turn {t+1}: guess={guess} | reward={env.reward}")
        for line in feedback.strip().splitlines():
            print("     " + line)
        messages += [{"role": "assistant", "content": generated},
                     {"role": "user", "content": feedback}]
    print(f"\nResult: reward={env.reward} | solved={env.done}\n")


play_wordle(WordleEnv(), fine_model, fine_tok)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

=== New game ===
[GAME] You are Playing Wordle.
A secret 5-letter word has been chosen. You have 6 attempts to guess it.
For each guess, wrap your word in sq ...

  Turn 1: guess=<thin | reward=0.0
     [GAME] [<thin]
     [GAME] You attempted an invalid move. Reason: You tried submitting a word in the wrong format. Please make sure to use squared brackets. Please resubmit a valid move and remember to follow the game rules to avoid penalties.
  Turn 2: guess=<thin | reward=0.0
     [GAME] [<thin]

Result: reward=0.0 | solved=True



## Summary

What you did:
1. Wrapped the local **Wordle** OpenEnv server in a tool-based `WordleEnv`.
2. Defined a single reward function (game outcome).
3. Trained **Qwen3-0.6B** with **GRPO** via TRL's `environment_factory` — an **agentic RL** loop where the model calls the `guess` tool, reads feedback, and acts again.
4. Evaluated the fine-tuned agent on fresh live games.

The takeaway: **OpenEnv environments are plug-ins.** Swap `WordleEnv` for any other OpenEnv environment (Catch, a coding env, a math grader) and the GRPO pipeline is unchanged.

### Tune it
- **Scale up:** raise `N_PROMPTS` (e.g. 200–1000).
- **Shaping:** add per-turn green/yellow rewards and a repetition penalty inside `WordleEnv.guess`.
- **Bigger model:** Qwen3-1.7B fits if you lower `vllm_gpu_memory_utilization` to ~0.15.
- **Track on the Hub:** enable `push_to_hub`, run `notebook_login()`.

### Troubleshooting
- `Failed to create MCP session` / `ConnectionClosedOK`: the Wordle server is down — `docker start textarena-wordle`.
- `OOM`: lower `vllm_gpu_memory_utilization` or `num_generations`, or reduce the dataset.
- `vLLM` import fails on Windows/Python 3.14: drop the `vllm_*` args and set `use_vllm=False` — the trainer falls back to transformers generation (slower but works).
